## Import libraries

In [2]:
import json
import os

import httpx
from dotenv import find_dotenv, load_dotenv
from pathlib import Path


### Load variables and URL

In [3]:
# Load environment variables
load_dotenv(find_dotenv(usecwd=True))

ura_key = os.getenv("URA_KEY")
ura_token = os.getenv("URA_TOKEN")

if not ura_key or not ura_token:
    raise RuntimeError(
        "URA_KEY and URA_TOKEN must be set (e.g. in .env). Run get_token.ipynb to refresh the token."
    )

# Save under project `data/` whether the kernel cwd is repo root or `notebook/`
DATA_DIR = Path.cwd() / "data"
if Path.cwd().name == "notebook":
    DATA_DIR = Path.cwd().parent / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

SERVICE = "PMI_Resi_Transaction"
BASE_URL = "https://eservice.ura.gov.sg/uraDataService/invokeUraDS/v1"
BATCHES = range(1, 5)  # batches 1 through 4


### Get all the data and save to json under data

In [4]:
headers = {
    "AccessKey": ura_key,
    "Token": ura_token,
    "Accept": "application/json",
    "User-Agent": "curl/8.7.1",
}

with httpx.Client(timeout=120.0) as client:
    for batch in BATCHES:
        url = f"{BASE_URL}?service={SERVICE}&batch={batch}"
        try:
            resp = client.get(url, headers=headers)
            resp.raise_for_status()
            raw = resp.text
        except httpx.HTTPStatusError as e:
            print(batch, e.response.status_code, e.response.reason_phrase)
            print(e.response.text)
            raise

        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            out_path = DATA_DIR / f"{SERVICE}_batch_{batch}.txt"
            out_path.write_text(raw, encoding="utf-8")
            print(batch, "non-JSON response saved to", out_path)
            continue

        out_path = DATA_DIR / f"{SERVICE}_batch_{batch}.json"
        out_path.write_text(json.dumps(data, indent=2), encoding="utf-8")
        status = data.get("Status") if isinstance(data, dict) else None
        print(batch, "->", out_path.name, f"Status={status}")


1 -> PMI_Resi_Transaction_batch_1.json Status=Success
2 -> PMI_Resi_Transaction_batch_2.json Status=Success
3 -> PMI_Resi_Transaction_batch_3.json Status=Success
4 -> PMI_Resi_Transaction_batch_4.json Status=Success
